In [8]:
import boto3
import time
import json

In [3]:
session = boto3.Session(
    region_name='us-east-1',
    aws_access_key_id='test',
    aws_secret_access_key='test'
)

endpoint = 'http://localhost:4566'

In [4]:
kinesis = session.client('kinesis', endpoint_url=endpoint)

In [5]:
events = [
    {"repo": "org/repo-a", "pr_number": 101, "author_type": "human", "ci_skipped": False},
    {"repo": "org/repo-a", "pr_number": 102, "author_type": "agent", "ci_skipped": True},
    {"repo": "org/repo-b", "pr_number": 55,  "author_type": "agent", "ci_skipped": True},
    {"repo": "org/repo-c", "pr_number": 12,  "author_type": "human", "ci_skipped": False},
    {"repo": "org/repo-a", "pr_number": 103, "author_type": "agent", "ci_skipped": False},
    {"repo": "org/repo-d", "pr_number": 8,   "author_type": "human", "ci_skipped": True},
]

In [10]:
for event in events:
    event['ingested_at'] = time.time()
    response = kinesis.put_record(
        StreamName='pr-events-stream',
        Data=json.dumps(event).encode('utf-8'),
        PartitionKey=event['repo']
    )
    print(event['repo'], event['pr_number'], '->', response['ShardId'], response['SequenceNumber'])

org/repo-a 101 -> shardId-000000000002 1795296782365
org/repo-a 102 -> shardId-000000000002 1795296782366
org/repo-b 55 -> shardId-000000000001 1795296782367
org/repo-c 12 -> shardId-000000000001 1795296782368
org/repo-a 103 -> shardId-000000000002 1795296782369
org/repo-d 8 -> shardId-000000000002 1795296782370


In [11]:
for event in events:
    event['ingested_at'] = time.time()
    response = kinesis.put_record(
        StreamName='pr-events-stream',
        Data=json.dumps(event).encode('utf-8'),
        PartitionKey=event['author_type']
    )
    print(event['repo'], event['author_type'], '->', response['ShardId'], response['SequenceNumber'])

org/repo-a human -> shardId-000000000001 1795296782371
org/repo-a agent -> shardId-000000000002 1795296782372
org/repo-b agent -> shardId-000000000002 1795296782373
org/repo-c human -> shardId-000000000001 1795296782374
org/repo-a agent -> shardId-000000000002 1795296782375
org/repo-d human -> shardId-000000000001 1795296782376


In [12]:
# send new events to test ShardIteratorType=LATESET

In [18]:
events = [
    {"repo": "org/repo-xxx", "pr_number": 23232, "author_type": "human", "ci_skipped": False},
    {"repo": "org/repo-yyy", "pr_number": 23232, "author_type": "agent", "ci_skipped": True},
    {"repo": "org/repo-zzz", "pr_number": 23232, "author_type": "agent", "ci_skipped": True},
    {"repo": "org/repo-xxx", "pr_number": 11111, "author_type": "agent", "ci_skipped": True},
]

In [19]:
for event in events:
    event['ingested_at'] = time.time()
    response = kinesis.put_record(
        StreamName='pr-events-stream',
        Data=json.dumps(event).encode('utf-8'),
        PartitionKey=event['repo']
    )
    print(event['repo'], event['pr_number'], '->', response['ShardId'], response['SequenceNumber'])

org/repo-xxx 23232 -> shardId-000000000002 1795296782383
org/repo-yyy 23232 -> shardId-000000000002 1795296782384
org/repo-zzz 23232 -> shardId-000000000002 1795296782385
org/repo-xxx 11111 -> shardId-000000000002 1795296782386
